# 03. Error Analysis & Threshold Tuning (Holdout Test Set)

**Project Goal:** Automated visual inspection of cast metal components (submersible pump impellers) to detect manufacturing defects.

> **Strict Constraint:** This analysis uses the **HOLD-OUT TEST SET ONLY** (`data/casting_data/casting_data/test/`). No training or validation data is evaluated here.

### Objectives:
1. Load the trained model (`models/model.keras`).
2. Run inference on all holdout test images and extract true labels & defect probabilities.
3. Plot the **Precision-Recall Curve** and generate a threshold trade-off table (thresholds 0.3 to 0.7).
4. Suggest an optimal decision threshold achieving **Recall >= 0.95** (crucial in manufacturing to prevent defective parts from reaching assembly).
5. Identify and isolate all misclassified test cases:
   - **False Negatives (FN):** Defective parts misclassified as OK (Critical risk).
   - **False Positives (FP):** Flawless parts flagged as defective (Scrap/re-inspection cost).
6. Generate **Grad-CAM visual explanation grids** for both error categories.
7. Export all figures to `reports/figures/`.

## 0. Colab Setup & Environment Verification
Run this section if executing in **Google Colab**. It pulls latest code, verifies dependencies, dataset files, and model weights.

In [ ]:
# Step 0a: Repository verification / pull
import os
import sys

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    repo_dir = 'vehicle-defect-detection'
    if not os.path.exists(repo_dir):
        !git clone https://github.com/annugupta25/vehicle-defect-detection.git
        %cd vehicle-defect-detection
    else:
        %cd vehicle-defect-detection
        !git pull origin main
    
    if '.' not in sys.path:
        sys.path.insert(0, '.')
    print("Working directory:", os.getcwd())
else:
    print("Running locally. Project root confirmed.")

In [ ]:
# Step 0b: Dependencies
if IN_COLAB:
    !pip install -q -r requirements.txt
    print("Dependencies verified!")
else:
    print("Local environment confirmed.")

In [ ]:
# Step 0c: Dataset verification
from src.utils import check_dataset_paths

status = check_dataset_paths("data/casting_data/casting_data")
print("Dataset path check:", status)

if not status["test_exists"] or status["test_def_count"] == 0:
    if IN_COLAB:
        from google.colab import files
        if not os.path.exists(os.path.expanduser("~/.kaggle/kaggle.json")):
            print("Upload kaggle.json:")
            files.upload()
            !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
        !kaggle datasets download -d ravirajsinh45/real-life-industrial-dataset-of-casting-product
        !unzip -q -o real-life-industrial-dataset-of-casting-product.zip -d data/
    else:
        raise FileNotFoundError("Test folder missing in data/casting_data/casting_data/test")

print("✓ Test dataset ready.")

In [ ]:
# Step 0d: Model file verification
model_path = "models/model.keras"
if not os.path.exists(model_path):
    if IN_COLAB:
        from google.colab import files
        print("Please upload your trained 'model.keras' file:")
        uploaded = files.upload()
        os.makedirs("models", exist_ok=True)
        for fn in uploaded.keys():
            if fn.endswith(".keras"):
                os.rename(fn, model_path)
                break
    else:
        raise FileNotFoundError(f"Trained model not found at: {model_path}")

print(f"✓ Trained model verified at {model_path} ({os.path.getsize(model_path)/(1024*1024):.2f} MB)")

## 1. Load Model & Holdout Test Dataset

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import cv2
from sklearn.metrics import (
    precision_recall_curve,
    confusion_matrix,
    classification_report,
    average_precision_score,
)
import tensorflow as tf
try:
    import keras
except ImportError:
    from tensorflow import keras

from src.gradcam import find_last_conv_layer, make_gradcam_heatmap, overlay_gradcam

# Directories
FIGURES_DIR = Path("reports/figures")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
TEST_DIR = Path("data/casting_data/casting_data/test")

# Load Model
try:
    model = keras.models.load_model(model_path)
except Exception:
    model = tf.keras.models.load_model(model_path)
print("Model architecture loaded successfully:")
print(f"  Name: {model.name}")
print(f"  Total parameters: {model.count_params():,}")

# Load Test Set using tf.keras utilities (batch=32, shuffle=False)
test_ds = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR,
    image_size=(224, 224),
    batch_size=32,
    color_mode="rgb",
    label_mode="binary",
    shuffle=False,
)
class_names = test_ds.class_names
test_file_paths = test_ds.file_paths

print(f"Test classes: {class_names} (0 = {class_names[0]}, 1 = {class_names[1]})")
print(f"Total holdout test samples: {len(test_file_paths)}")

## 2. Model Inference & Prediction Extraction

Extract true labels and model predicted probabilities on the entire holdout test set.
- `class 0`: `def_front` (Defective casting)
- `class 1`: `ok_front` (Flawless casting)
- Sigmoid output $\hat{y} = P(\text{OK})$.
- Defect probability $P(\text{Defective}) = 1.0 - \hat{y}$.

In [ ]:
y_true_list = []
y_ok_probs_list = []

for images, labels in test_ds:
    preds = model.predict(images, verbose=0)
    y_ok_probs_list.append(preds.flatten())
    y_true_list.append(labels.numpy().flatten())

y_true = np.concatenate(y_true_list).astype(int)
y_ok_probs = np.concatenate(y_ok_probs_list)

# Binary defect ground truth: 1 if Defective, 0 if OK
# In test_ds, label 0 is def_front. Let's define binary positive = Defective
y_true_defect = (y_true == 0).astype(int)
y_defect_prob = 1.0 - y_ok_probs

print(f"Evaluated {len(y_true)} test images:")
print(f"  - Defective parts: {np.sum(y_true_defect)}")
print(f"  - OK parts:        {len(y_true_defect) - np.sum(y_true_defect)}")

## 3. Threshold Trade-Off Analysis & Precision-Recall Curve

In manufacturing quality control:
- **Recall (Defect):** $\frac{\text{TP}}{\text{TP} + \text{FN}}$ = Percentage of defective parts caught.
- **Precision (Defect):** $\frac{\text{TP}}{\text{TP} + \text{FP}}$ = Percentage of flagged parts that are genuinely defective.

In [ ]:
# Table of Precision and Recall across thresholds from 0.30 to 0.70
threshold_values = [0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70]
tradeoff_records = []

for t in threshold_values:
    pred_defect_at_t = (y_defect_prob >= t).astype(int)
    tp = np.sum((pred_defect_at_t == 1) & (y_true_defect == 1))
    fp = np.sum((pred_defect_at_t == 1) & (y_true_defect == 0))
    fn = np.sum((pred_defect_at_t == 0) & (y_true_defect == 1))
    tn = np.sum((pred_defect_at_t == 0) & (y_true_defect == 0))
    
    rec = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    f1 = (2 * prec * rec) / (prec + rec) if (prec + rec) > 0 else 0.0
    
    tradeoff_records.append({
        "Threshold (P_defect)": round(t, 2),
        "Precision (Defect)": round(prec, 4),
        "Recall (Defect)": round(rec, 4),
        "F1-Score": round(f1, 4),
        "True Positives (TP)": int(tp),
        "False Negatives (FN)": int(fn),
        "False Positives (FP)": int(fp),
        "Recall >= 0.95": "YES " if rec >= 0.95 else "NO ❌",
    })

df_tradeoff = pd.DataFrame(tradeoff_records)
print("=== PRECISION / RECALL THRESHOLD TRADE-OFF TABLE ===")
display(df_tradeoff)

# Scan fine-grained thresholds to suggest optimal threshold for Recall >= 0.95
fine_thresholds = np.linspace(0.10, 0.90, 81)
best_t = None
best_prec = -1.0
best_rec = -1.0

for t in fine_thresholds:
    pred_d = (y_defect_prob >= t).astype(int)
    tp = np.sum((pred_d == 1) & (y_true_defect == 1))
    fp = np.sum((pred_d == 1) & (y_true_defect == 0))
    fn = np.sum((pred_d == 0) & (y_true_defect == 1))
    rec = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    
    if rec >= 0.95 and prec > best_prec:
        best_prec = prec
        best_rec = rec
        best_t = round(float(t), 2)

if best_t is None:
    best_t = 0.50

print(f"\n RECOMMENDED DECISION THRESHOLD: {best_t:.2f}")
print(f"   Yields Defect Recall: {best_rec:.2%}, Defect Precision: {best_prec:.2%}")

In [ ]:
# Plot Precision-Recall Curve
precisions, recalls, pr_thresholds = precision_recall_curve(y_true_defect, y_defect_prob)
ap_score = average_precision_score(y_true_defect, y_defect_prob)

plt.figure(figsize=(8, 6))
plt.plot(recalls, precisions, color="#2b5c8f", lw=2.5, label=f"PR Curve (AP = {ap_score:.4f})")
plt.axvline(0.95, color="#d9534f", linestyle="--", lw=1.5, label="Target Recall >= 0.95")

# Mark recommended threshold
if best_t is not None:
    plt.scatter([best_rec], [best_prec], color="#d9534f", s=120, zorder=5, 
                label=f"Suggested T={best_t:.2f} (R={best_rec:.1%}, P={best_prec:.1%})")

plt.title("Precision-Recall Curve (Defect Detection — Holdout Test Set)", fontsize=13, fontweight="bold", pad=12)
plt.xlabel("Recall (Defective)", fontsize=11)
plt.ylabel("Precision (Defective)", fontsize=11)
plt.xlim([0.0, 1.02])
plt.ylim([0.0, 1.05])
plt.legend(loc="lower left", fontsize=10.5)
plt.grid(True, alpha=0.3)

pr_fig_path = FIGURES_DIR / "precision_recall_curve.png"
plt.savefig(pr_fig_path, dpi=300, bbox_inches="tight")
print(f"Precision-Recall curve saved to: {pr_fig_path}")
plt.show()

## 4. Misclassified Test Cases Identification

Using the selected decision threshold, we classify test images into:
1. **False Negatives:** Ground truth is Defective, but model predicted OK.
2. **False Positives:** Ground truth is OK, but model predicted Defective.

In [ ]:
eval_threshold = best_t
predicted_defect = (y_defect_prob >= eval_threshold).astype(int)

false_negatives_indices = np.where((y_true_defect == 1) & (predicted_defect == 0))[0]
false_positives_indices = np.where((y_true_defect == 0) & (predicted_defect == 1))[0]

print(f"At Threshold T = {eval_threshold:.2f}:")
print(f"  False Negatives (Defective missed): {len(false_negatives_indices)}")
print(f"  False Positives (False alarms):    {len(false_positives_indices)}")

## 5. False Negatives with Grad-CAM Overlays

Visual inspection of missed defects: Grad-CAM explains where the model focused when it failed to flag the defect.

In [ ]:
target_conv = find_last_conv_layer(model)
print(f"Grad-CAM target conv layer: {target_conv}")

if len(false_negatives_indices) > 0:
    n_show = min(len(false_negatives_indices), 6)
    fig, axes = plt.subplots(n_show, 2, figsize=(10, 4 * n_show))
    if n_show == 1:
        axes = np.expand_dims(axes, 0)
        
    for i, idx in enumerate(false_negatives_indices[:n_show]):
        img_path = test_file_paths[idx]
        pil_img = Image.open(img_path).convert("RGB").resize((224, 224))
        img_arr = np.expand_dims(np.array(pil_img, dtype=np.float32), 0)
        uint8_arr = np.array(pil_img, dtype=np.uint8)
        
        # Generate Grad-CAM
        heatmap = make_gradcam_heatmap(img_arr, model, last_conv_layer_name=target_conv, pred_index=0)
        overlay, _ = overlay_gradcam(uint8_arr, heatmap, alpha=0.45)
        
        p_def = y_defect_prob[idx]
        p_ok = y_ok_probs[idx]
        fname = Path(img_path).name
        
        axes[i, 0].imshow(uint8_arr)
        axes[i, 0].set_title(f"FN #{i+1}: {fname}\nTrue: Defective | Pred: OK (P_ok: {p_ok:.1%})", fontsize=10, color="#d9534f")
        axes[i, 0].axis("off")
        
        axes[i, 1].imshow(overlay)
        axes[i, 1].set_title(f"Grad-CAM Defect Attention\nP(Defect) = {p_def:.1%}", fontsize=10)
        axes[i, 1].axis("off")
        
    plt.suptitle("False Negatives (Missed Defects) with Grad-CAM Overlays", fontsize=14, fontweight="bold", y=1.01)
    plt.tight_layout()
    fn_fig_path = FIGURES_DIR / "false_negatives_gradcam.png"
    plt.savefig(fn_fig_path, dpi=300, bbox_inches="tight")
    print(f"False Negatives grid saved to: {fn_fig_path}")
    plt.show()
else:
    print("🎉 No False Negatives found at this threshold! (100% Defect Recall)")

## 6. False Positives with Grad-CAM Overlays

Visual inspection of false alarms: Grad-CAM highlights what surface feature triggered the false defect warning.

In [ ]:
if len(false_positives_indices) > 0:
    n_show = min(len(false_positives_indices), 6)
    fig, axes = plt.subplots(n_show, 2, figsize=(10, 4 * n_show))
    if n_show == 1:
        axes = np.expand_dims(axes, 0)
        
    for i, idx in enumerate(false_positives_indices[:n_show]):
        img_path = test_file_paths[idx]
        pil_img = Image.open(img_path).convert("RGB").resize((224, 224))
        img_arr = np.expand_dims(np.array(pil_img, dtype=np.float32), 0)
        uint8_arr = np.array(pil_img, dtype=np.uint8)
        
        # Generate Grad-CAM highlighting defect activation
        heatmap = make_gradcam_heatmap(img_arr, model, last_conv_layer_name=target_conv, pred_index=0)
        overlay, _ = overlay_gradcam(uint8_arr, heatmap, alpha=0.45)
        
        p_def = y_defect_prob[idx]
        fname = Path(img_path).name
        
        axes[i, 0].imshow(uint8_arr)
        axes[i, 0].set_title(f"FP #{i+1}: {fname}\nTrue: OK | Pred: Defective (P_def: {p_def:.1%})", fontsize=10, color="#f0ad4e")
        axes[i, 0].axis("off")
        
        axes[i, 1].imshow(overlay)
        axes[i, 1].set_title(f"Grad-CAM Trigger Region\nP(Defect) = {p_def:.1%}", fontsize=10)
        axes[i, 1].axis("off")
        
    plt.suptitle("False Positives (False Alarms) with Grad-CAM Overlays", fontsize=14, fontweight="bold", y=1.01)
    plt.tight_layout()
    fp_fig_path = FIGURES_DIR / "false_positives_gradcam.png"
    plt.savefig(fp_fig_path, dpi=300, bbox_inches="tight")
    print(f"False Positives grid saved to: {fp_fig_path}")
    plt.show()
else:
    print("🎉 No False Positives found at this threshold!")

## 7. Key Findings & Threshold Recommendations

### Manufacturing Takeaways:
1. **Asymmetric Risk:** A False Negative (shipping a cracked impeller to vehicle assembly) carries severe warranty and safety liability. A False Positive (re-inspecting a flawless part) only costs secondary optical inspection time.
2. **Optimal Operating Point:** Setting the decision threshold based on Recall >= 0.95 ensures maximum defect capture.
3. **Streamlit Deployment:** The threshold slider in `app/app.py` allows factory line supervisors to dynamically adjust sensitivity depending on production shifts and batch tolerance requirements.